In [4]:
import pandas as pd
import numpy as np

print("✅ Libraries loaded successfully")




✅ Libraries loaded successfully


In [3]:
file_path = "../data/raw/scanner_data.csv"

# Load only first 100,000 rows for testing
df_sample = pd.read_csv(file_path, nrows=100000)

print("Sample shape:", df_sample.shape)
df_sample.head()


Sample shape: (100000, 8)


,Unnamed: 0,Date,Customer_ID,Transaction_ID,SKU_Category,SKU,Quantity,Sales_Amount
0,1,02/01/2016,2547,1,X52,0EM7L,1.0,3.13
1,2,02/01/2016,822,2,2ML,68BRQ,1.0,5.46
2,3,02/01/2016,3686,3,0H2,CZUZX,1.0,6.35
3,4,02/01/2016,3719,4,0H2,549KK,1.0,5.59
4,5,02/01/2016,9200,5,0H2,K8EHH,1.0,6.88


#Step 3: Count Total Rows (Full Dataset)

In [5]:
# count total rows in file
with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
    total_rows = sum(1 for line in f) - 1  # subtract header
print("Total rows in dataset:", total_rows)


Total rows in dataset: 131706


#Step 4: Inspect Sample

In [7]:
# Info about columns
df_sample.info(memory_usage="deep")

# Missing values
df_sample.isna().sum().sort_values(ascending=False).head(10)

# Duplicates
print("Duplicates in sample:", df_sample.duplicated().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Unnamed: 0      100000 non-null  int64  
 1   Date            100000 non-null  object 
 2   Customer_ID     100000 non-null  int64  
 3   Transaction_ID  100000 non-null  int64  
 4   SKU_Category    100000 non-null  object 
 5   SKU             100000 non-null  object 
 6   Quantity        100000 non-null  float64
 7   Sales_Amount    100000 non-null  float64
dtypes: float64(2), int64(3), object(3)
memory usage: 19.6 MB
Duplicates in sample: 0


#Cleaning Large Dataset

#Handle Missing Values

In [9]:
import pandas as pd

file_path = "../data/raw/scanner_data.csv"

chunk_size = 200000   # process 200k rows at a time
cleaned_chunks = []

for chunk in pd.read_csv(file_path, chunksize=chunk_size):
    # 1. Drop completely empty rows
    chunk = chunk.dropna(how="all")
    
    # 2. Fill numeric missing with median
    for col in chunk.select_dtypes(include="number").columns:
        median_val = chunk[col].median()
        chunk[col] = chunk[col].fillna(median_val)
    
    # 3. Fill text missing with "Unknown"
    for col in chunk.select_dtypes(include="object").columns:
        chunk[col] = chunk[col].fillna("Unknown")
    
    cleaned_chunks.append(chunk)

# Combine back into one dataframe (be careful: may use memory if huge)
df_clean = pd.concat(cleaned_chunks, ignore_index=True)

print("✅ Missing values handled")
df_clean.head()


✅ Missing values handled


,Unnamed: 0,Date,Customer_ID,Transaction_ID,SKU_Category,SKU,Quantity,Sales_Amount
0,1,02/01/2016,2547,1,X52,0EM7L,1.0,3.13
1,2,02/01/2016,822,2,2ML,68BRQ,1.0,5.46
2,3,02/01/2016,3686,3,0H2,CZUZX,1.0,6.35
3,4,02/01/2016,3719,4,0H2,549KK,1.0,5.59
4,5,02/01/2016,9200,5,0H2,K8EHH,1.0,6.88


#Remove Duplicates

In [10]:
before = df_clean.shape[0]
df_clean = df_clean.drop_duplicates()
after = df_clean.shape[0]

print(f"Removed {before - after} duplicate rows")


Removed 0 duplicate rows


#Fix Data Types (Optimize Memory)

In [11]:
def optimize_dtypes(df):
    for col in df.select_dtypes(include="int").columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    for col in df.select_dtypes(include="float").columns:
        df[col] = pd.to_numeric(df[col], downcast="float")
    return df

df_clean = optimize_dtypes(df_clean)

print("✅ Data types optimized")
df_clean.info(memory_usage="deep")


✅ Data types optimized
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131706 entries, 0 to 131705
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Unnamed: 0      131706 non-null  int32  
 1   Date            131706 non-null  object 
 2   Customer_ID     131706 non-null  int16  
 3   Transaction_ID  131706 non-null  int32  
 4   SKU_Category    131706 non-null  object 
 5   SKU             131706 non-null  object 
 6   Quantity        131706 non-null  float32
 7   Sales_Amount    131706 non-null  float32
dtypes: float32(2), int16(1), int32(2), object(3)
memory usage: 23.0 MB


#Save Cleaned Data (Parquet for Speed )

In [12]:
df_clean.to_parquet("../data/processed/retail_clean.parquet", index=False)
print("✅ Cleaned dataset saved as Parquet")


✅ Cleaned dataset saved as Parquet


#Reload Quickly

In [13]:
df = pd.read_parquet("../data/processed/retail_clean.parquet")
df.head()


,Unnamed: 0,Date,Customer_ID,Transaction_ID,SKU_Category,SKU,Quantity,Sales_Amount
0,1,02/01/2016,2547,1,X52,0EM7L,1.0,3.13
1,2,02/01/2016,822,2,2ML,68BRQ,1.0,5.46
2,3,02/01/2016,3686,3,0H2,CZUZX,1.0,6.35
3,4,02/01/2016,3719,4,0H2,549KK,1.0,5.59
4,5,02/01/2016,9200,5,0H2,K8EHH,1.0,6.88
